# Eksperiment 3-1 - ELO rangovi + SMA(5) forma

In [1]:
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from experiment_utils import run_experiment

import warnings
warnings.filterwarnings('ignore')

## Podaci

In [2]:
df = pd.read_csv('../data/Processed_Project_Data/fifa_and_elo_rankings_clean_filled.csv')
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df['date'] = pd.to_datetime(df['date'], format='mixed')
df = df.sort_values('date').reset_index(drop=True)

print(f'Ukupno meceva: {len(df)}')
print(df['result'].value_counts())

Ukupno meceva: 1239
result
 1    582
-1    371
 0    286
Name: count, dtype: int64


## SMA(5)

In [3]:
home_rows = df[['date', 'home_team', 'result']].rename(columns={'home_team': 'team', 'result': 'team_result'})
away_rows = df[['date', 'away_team', 'result']].copy()
away_rows['team_result'] = -away_rows['result']
away_rows = away_rows[['date', 'away_team', 'team_result']].rename(columns={'away_team': 'team'})

team_results = pd.concat([home_rows, away_rows]).sort_values('date').reset_index(drop=True)
team_results['sma5'] = (
    team_results.groupby('team')['team_result']
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

df = df.merge(team_results.rename(columns={'team': 'home_team', 'sma5': 'home_form_sma5'})[['date', 'home_team', 'home_form_sma5']], on=['date', 'home_team'], how='left')
df = df.merge(team_results.rename(columns={'team': 'away_team', 'sma5': 'away_form_sma5'})[['date', 'away_team', 'away_form_sma5']], on=['date', 'away_team'], how='left')
df = df.dropna(subset=['home_form_sma5', 'away_form_sma5', 'home_elo', 'away_elo']).reset_index(drop=True)

print(f'Meceva nakon filtriranja: {len(df)}')

Meceva nakon filtriranja: 614


## Priprema i pokretanje

In [4]:
X = df[['home_elo', 'away_elo', 'home_form_sma5', 'away_form_sma5']].values
le = LabelEncoder()
y = le.fit_transform(df['result'].values)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_sc, dtype=torch.float32)
y_train_t = torch.tensor(y_train,    dtype=torch.long)
X_test_t  = torch.tensor(X_test_sc,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,     dtype=torch.long)

run_experiment(X_train_t, y_train_t, X_test_t, y_test_t)

Run 1/10
Accuracy: 0.5528

Run 2/10
Accuracy: 0.5772

Run 3/10
Accuracy: 0.5772

Run 4/10
Accuracy: 0.5772

Run 5/10
Accuracy: 0.6098

Run 6/10
Accuracy: 0.5772

Run 7/10
Accuracy: 0.5691

Run 8/10
Accuracy: 0.6260

Run 9/10
Accuracy: 0.5772

Run 10/10
Accuracy: 0.5691

Mean accuracy: 0.5813
Classification report for last run:
              precision    recall  f1-score   support

    Away Win       0.54      0.58      0.56        33
        Draw       0.33      0.16      0.22        31
    Home Win       0.63      0.78      0.70        59

    accuracy                           0.57       123
   macro avg       0.50      0.51      0.49       123
weighted avg       0.53      0.57      0.54       123

Confusion matrix for last run:
               Pred Away Win  Pred Draw  Pred Home Win
True Away Win             19          3             11
True Draw                 10          5             16
True Home Win              6          7             46
